In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import gc

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

In [3]:
class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 1024)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.2)
        self.layer2 = nn.Linear(1024, 1024)
        self.relu2 = nn.ReLU()
        self.classifier = nn.Linear(1024, 10)

    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = self.relu1(self.layer1(x))
        residual = x
        x = self.drop1(x)
        x = self.relu2(self.layer2(x))
        x = x + residual
        return self.classifier(x)
model = mlp()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())
gc.collect()

0

In [4]:
epochs = 50
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

In [5]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)
    
    gc.collect()

Epoch [1/50]
Training loss 0.19503835126459598
Training accuracy 94.02166666666666 %
Testing loss 0.10630691424012184
Testing accuracy 96.69 %
Epoch [2/50]
Training loss 0.08894946458886067
Training accuracy 97.21166666666667 %
Testing loss 0.09428046904504299
Testing accuracy 97.23 %
Epoch [3/50]
Training loss 0.06502771272137761
Training accuracy 97.965 %
Testing loss 0.10692987479269504
Testing accuracy 96.87 %
Epoch [4/50]
Training loss 0.05057580908052623
Training accuracy 98.405 %
Testing loss 0.08639323189854622
Testing accuracy 97.68 %
Epoch [5/50]
Training loss 0.042929947349755096
Training accuracy 98.635 %
Testing loss 0.09655748065561057
Testing accuracy 97.31 %
Epoch [6/50]
Training loss 0.03545409074588679
Training accuracy 98.865 %
Testing loss 0.09535720320418477
Testing accuracy 97.55 %
Epoch [7/50]
Training loss 0.02906123645976768
Training accuracy 99.105 %
Testing loss 0.10251627117395401
Testing accuracy 97.44 %
Epoch [8/50]
Training loss 0.028962331986722226
Train